# SFT Distillation: Training Gemma4 on Claude Reasoning Traces

**Author:** Behrooz Azarkhalili
**Last updated:** 2026-04

---

## What You'll Learn

This notebook demonstrates **reasoning distillation** via Supervised Fine-Tuning (SFT) --
transferring the chain-of-thought reasoning capabilities of large frontier models
(Claude Opus 4 / Sonnet 4) into Liquid Foundation Model 2.5 (Gemma4-E2B-it),
a compact state-space model designed for efficient deployment.

| Step | Description |
|------|-------------|
| 1. Dataset | Load Claude reasoning traces (`ermiaazarkhalili/claude-reasoning-distillation`) |
| 2. Adapter | Embed the `thinking` field as `<think>...</think>` blocks in ChatML |
| 3. Training | QLoRA fine-tuning with TRL's `SFTTrainer` |
| 4. Inference | Generate reasoning traces and detect `<think>` blocks |
| 5. Export | Merge adapter, convert to GGUF, push to Hub |

### Hardware Requirements

- **Minimum:** 16 GB VRAM (e.g. NVIDIA T4, RTX 4090, A10G)
- **Recommended:** 24+ GB VRAM for longer sequences
- **SLURM:** Works on any partition with at least one GPU

### Key Benefits

- **Reasoning distillation** from Claude Opus/Sonnet into a tiny open model
- **100% thinking traces** -- every training sample includes chain-of-thought
- **State-space architecture** -- Gemma4 uses a hybrid transformer/attention design for fast inference
- **QLoRA** keeps memory footprint under 10 GB for the E2B model
- **`<think>` blocks** -- clean, parseable reasoning format in the output

### Why Gemma4?

google's Gemma4 is a hybrid state-space model that combines the efficiency of transformers
with selective attention layers. At E2B parameters, it offers competitive performance
with significantly faster inference than pure-transformer models of similar size.
Unlike Qwen3.5 or Gemma4, Gemma4 does not have native thinking tokens -- we embed
reasoning as `<think>...</think>` XML blocks directly in the assistant content.

In [ ]:
# ============================================================================
# Install Dependencies
# ============================================================================
# Colab / bare-metal -- uncomment the pip line below:
# !pip install -q transformers trl peft accelerate bitsandbytes datasets

# SLURM (Digital Research Alliance / Fir cluster):
#   module load gcc arrow python/3.11.5
#   source /scratch/$USER/venvs/hf_trl/bin/activate
# All packages are pre-installed in the venv -- skip the pip install above.

In [ ]:
# ============================================================================
# Imports & GPU Detection
# ============================================================================
from __future__ import annotations

import os
import re
import statistics
import time
import warnings
from dataclasses import dataclass, field
from typing import Any

import torch
import transformers
import trl
from datasets import load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------------
# GPU Detection
# ---------------------------------------------------------------------------
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[OK] GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("[WARN] No GPU detected -- training will be extremely slow")

print(f"[OK] PyTorch {torch.__version__}")
print(f"[OK] Transformers {transformers.__version__}")
print(f"[OK] TRL {trl.__version__}")
print(f"[OK] CUDA {torch.version.cuda}")

In [ ]:
# ============================================================================
# Hugging Face Authentication
# ============================================================================
# Option 1: .env file (recommended for SLURM / local)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("[OK] Loaded .env file")
except ImportError:
    pass

# Option 2: Colab secrets
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
    print("[OK] Loaded Colab secret")
except (ImportError, Exception):
    pass

# Verify token is available
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("[OK] Authenticated with Hugging Face Hub")
else:
    print("[WARN] No HF_TOKEN found -- private datasets / Hub push will fail")
    print("       Set HF_TOKEN in your environment or .env file")

---

## 1. Configuration

All hyperparameters are grouped into **dataclasses** for reproducibility and easy
experimentation. Toggle `SMOKE_TEST = False` to run a quick 5-step sanity check
before committing to a full training run.

- **`ModelConfig`** -- model identifier, quantization, sequence length
- **`LoRAConfig`** -- adapter rank, alpha, target modules
- **`TrainingConfig`** -- optimizer, scheduler, batch size, epochs

In [ ]:
# ============================================================================
# Configuration Dataclasses
# ============================================================================

SMOKE_TEST: bool = False  # Set False for full training


@dataclass
class ModelConfig:
    """Model and quantization settings for Gemma4."""

    model_name: str = "google/gemma-4-E2B-it"
    hub_model_id: str = "ermiaazarkhalili/Gemma4-E2B-SFT-Claude-Reasoning"
    max_length: int = 2048
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    bnb_4bit_compute_dtype: str = "bfloat16"
    use_double_quant: bool = True
    attn_implementation: str = "sdpa"  # Gemma4 may not support sdpa/flash_attention_2


@dataclass
class LoRAConfig:
    """LoRA adapter settings."""

    r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    # Target modules are discovered dynamically from the model --
    # see create_lora_config() below. These are typical transformer names
    # that may or may not exist in Gemma4's hybrid transformer architecture.
    # Gemma4 uses Gemma4ClippableLinear (wraps nn.Linear). PEFT 0.18.x
    # doesn't recognize it. "all-linear" recursively finds nested nn.Linear.
    # Approach from PEFT PR #3136 and community workarounds.
    target_modules: str = "all-linear"
    exclude_modules: list[str] = field(default_factory=lambda: [
        "vision_tower", "multi_modal_projector", "audio_tower",
    ])
    bias: str = "none"
    task_type: str = "CAUSAL_LM"


@dataclass
class TrainingConfig:
    """Training hyperparameters."""

    output_dir: str = "./gemma4-sft-claude-reasoning"
    num_train_epochs: int = 1
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-5
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    bf16: bool = True
    logging_steps: int = 10
    save_strategy: str = "no"
    report_to: str = "none"
    push_to_hub: bool = False  # We push manually after merge
    seed: int = 42
    dataset_name: str = "ermiaazarkhalili/claude-reasoning-distillation"
    dataset_config: str = "sft"


# ---- Instantiate ----
model_cfg = ModelConfig()
lora_cfg = LoRAConfig()
train_cfg = TrainingConfig()

# ---- Smoke-test overrides ----
if SMOKE_TEST:
    train_cfg.logging_steps = 1
    train_cfg.push_to_hub = False
    print("[SMOKE] Smoke-test mode: max_steps=5, push_to_hub=False")
else:
    print(f"[TRAIN] Full training: {train_cfg.num_train_epochs} epoch(s)")

print(f"[OK] Model:    {model_cfg.model_name}")
print(f"[OK] Hub ID:   {model_cfg.hub_model_id}")
print(f"[OK] LoRA r:   {lora_cfg.r}, alpha: {lora_cfg.lora_alpha}")
print(f"[OK] Epochs:   {train_cfg.num_train_epochs}")
print(f"[OK] LR:       {train_cfg.learning_rate}")
print(f"[OK] Batch:    {train_cfg.per_device_train_batch_size} x {train_cfg.gradient_accumulation_steps} = {train_cfg.per_device_train_batch_size * train_cfg.gradient_accumulation_steps}")

In [ ]:
# ============================================================================
# Hardware-Aware Configuration
# ============================================================================

def setup_hardware_config(
    model_cfg: ModelConfig,
    train_cfg: TrainingConfig,
) -> BitsAndBytesConfig:
    """Create quantization config and adjust batch size for available VRAM.

    Args:
        model_cfg: Model configuration dataclass.
        train_cfg: Training configuration dataclass (modified in-place).

    Returns:
        BitsAndBytesConfig for 4-bit QLoRA loading.
    """
    compute_dtype = getattr(torch, model_cfg.bnb_4bit_compute_dtype)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=model_cfg.load_in_4bit,
        bnb_4bit_quant_type=model_cfg.bnb_4bit_quant_type,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=model_cfg.use_double_quant,
    )

    # Auto-adjust batch size and precision based on GPU memory
    if torch.cuda.is_available():
        capability = torch.cuda.get_device_capability()
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

        if capability[0] >= 8:  # Ampere+ (A100, H100, RTX 40xx)
            train_cfg.bf16 = True
            print(f"[HW] {gpu_name}: bf16 enabled (compute {capability[0]}.{capability[1]})")
        else:
            train_cfg.bf16 = False
            print(f"[HW] {gpu_name}: fp16 fallback (compute {capability[0]}.{capability[1]})")

        if gpu_mem_gb < 16:
            train_cfg.per_device_train_batch_size = 1
            train_cfg.gradient_accumulation_steps = 8
            print(f"[HW] Low VRAM ({gpu_mem_gb:.0f} GB) -- batch=1, grad_accum=8")
        elif gpu_mem_gb >= 40:
            train_cfg.per_device_train_batch_size = 4
            train_cfg.gradient_accumulation_steps = 2
            print(f"[HW] High VRAM ({gpu_mem_gb:.0f} GB) -- batch=4, grad_accum=2")
        else:
            print(f"[HW] Medium VRAM ({gpu_mem_gb:.0f} GB) -- batch={train_cfg.per_device_train_batch_size}, grad_accum={train_cfg.gradient_accumulation_steps}")

        # SLURM detection
        if os.environ.get("SLURM_JOB_ID"):
            print(f"[HW] SLURM job {os.environ['SLURM_JOB_ID']} detected")
    else:
        train_cfg.bf16 = False
        print("[HW] CPU-only -- no mixed precision")

    print(f"[OK] Quantization: {model_cfg.bnb_4bit_quant_type}, compute_dtype={model_cfg.bnb_4bit_compute_dtype}")
    return bnb_config


bnb_config = setup_hardware_config(model_cfg, train_cfg)
print(f"[OK] Effective batch size: {train_cfg.per_device_train_batch_size * train_cfg.gradient_accumulation_steps}")

---

## 2. Dataset: Claude Reasoning Distillation

The dataset contains multi-turn conversations with **thinking traces** generated by
Claude Opus 4 and Sonnet 4. Each assistant message includes a `thinking` field with
the model's chain-of-thought reasoning.

- **Source:** `ermiaazarkhalili/claude-reasoning-distillation` (private, config=`"sft"`)
- **Train:** 10,477 samples | **Test:** 549 samples
- **100% thinking traces** -- every sample includes chain-of-thought

**Schema** (each sample):
```json
{
  "messages": [
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "...", "thinking": "step-by-step reasoning..."}
  ]
}
```

The `thinking` field is the key differentiator -- it captures *how* Claude reasons,
not just *what* it outputs.

In [ ]:
# ============================================================================
# Load Dataset
# ============================================================================
print(f"[....] Loading {train_cfg.dataset_name} (config={train_cfg.dataset_config!r}) ...")

train_dataset = load_dataset(train_cfg.dataset_name, train_cfg.dataset_config, split="train")
test_dataset = load_dataset(train_cfg.dataset_name, train_cfg.dataset_config, split="test")

print(f"[OK] Train: {len(train_dataset):,} samples")
print(f"[OK] Test:  {len(test_dataset):,} samples")
print(f"[OK] Columns: {train_dataset.column_names}")

# ---------------------------------------------------------------------------
# Dataset statistics
# ---------------------------------------------------------------------------
def compute_dataset_stats(dataset: Any) -> dict[str, Any]:
    """Compute basic statistics about the dataset.

    Args:
        dataset: HF dataset with 'messages' column.

    Returns:
        Dictionary of statistics.
    """
    n_turns: list[int] = []
    has_thinking: int = 0
    total_assistant_turns: int = 0
    thinking_turns: int = 0

    for sample in dataset:
        msgs = sample["messages"]
        n_turns.append(len(msgs))
        for msg in msgs:
            if msg["role"] == "assistant":
                total_assistant_turns += 1
                if msg.get("thinking"):
                    thinking_turns += 1
        if any(msg.get("thinking") for msg in msgs):
            has_thinking += 1

    return {
        "total_samples": len(dataset),
        "avg_turns": sum(n_turns) / len(n_turns),
        "max_turns": max(n_turns),
        "min_turns": min(n_turns),
        "assistant_turns": total_assistant_turns,
        "thinking_turns": thinking_turns,
        "pct_thinking": thinking_turns / max(1, total_assistant_turns) * 100,
    }


stats = compute_dataset_stats(train_dataset)
print("\n--- Train Set Statistics ---")
for k, v in stats.items():
    print(f"  {k}: {v:.1f}" if isinstance(v, float) else f"  {k}: {v}")

# ---------------------------------------------------------------------------
# Preview a sample
# ---------------------------------------------------------------------------
sample = train_dataset[0]
print("\n--- Sample 0 ---")
for msg in sample["messages"]:
    role = msg["role"]
    content = (msg.get("content") or "")[:120]
    thinking = msg.get("thinking")
    print(f"  [{role}] {content}{'...' if len(msg.get('content', '') or '') > 120 else ''}")
    if thinking:
        print(f"    [thinking] {thinking[:80]}...")

---

## 3. Thinking-Field Adapter: Gemma4

Gemma4 uses a standard Gemma4's native thinking mode template and does **not** have native thinking tokens
(unlike Qwen3.5's `reasoning_content` or Gemma4's `<|think|>` trigger). Our adapter
strategy embeds the reasoning trace directly into the assistant content:

```
<|think|>
{thinking field content -- step-by-step reasoning}
</think>
{final answer content}
```

This approach:
- Preserves the full chain-of-thought in a parseable XML block
- Works with any Gemma4's native thinking mode-compatible model
- At inference time, we can detect and extract `<|think|>...</think>` blocks
- The model learns to generate the `<|think|>` prefix before answering

> Unlike Qwen3.5 which uses a `reasoning_content` field or Gemma4 which uses
> `<|think|>` system prompt trigger, Gemma4 relies on explicit XML tags embedded
> in the content itself.

In [ ]:
# ============================================================================
# Thinking-Field Adapter for Gemma4
# ============================================================================

def adapt_for_gemma4(sample: dict[str, Any], tokenizer: AutoTokenizer) -> dict[str, str]:
    """Embed the thinking field as <think>...</think> in assistant content.

    Args:
        sample: A dataset row with a "messages" list. Each assistant message
            may have an optional "thinking" key.
        tokenizer: The Gemma4 tokenizer (used for apply_chat_template).

    Returns:
        Dictionary with a single "text" key containing the formatted chat.
    """
    messages: list[dict[str, str]] = []

    for msg in sample["messages"]:
        m = dict(msg)

        if m["role"] == "assistant":
            thinking = m.pop("thinking", None)
            if thinking:
                m["content"] = f"<think>{thinking}</think>\n{m['content']}"
            else:
                m.pop("thinking", None)
        else:
            # Strip thinking from non-assistant messages (shouldn't exist, but be safe)
            m.pop("thinking", None)

        messages.append(m)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


print("[OK] adapt_for_gemma4() defined")
print("     Strategy: <think>{thinking}</think>\\n{content}")

In [ ]:
# ============================================================================
# Load Tokenizer & Apply Adapter
# ============================================================================
MODEL_NAME: str = model_cfg.model_name

print(f"[....] Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Ensure pad token exists (Gemma4 may not set one)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"[OK] Set pad_token = eos_token ({tokenizer.pad_token!r})")

print(f"[OK] Vocab size: {tokenizer.vocab_size:,}")
print(f"[OK] eos_token:  {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"[OK] pad_token:  {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

# ---------------------------------------------------------------------------
# Apply adapter to datasets
# ---------------------------------------------------------------------------
print(f"\n[....] Applying Gemma4 adapter to datasets ...")

def _apply_adapter(sample: dict) -> dict[str, str]:
    return adapt_for_gemma4(sample, tokenizer)

train_adapted = train_dataset.map(
    _apply_adapter,
    remove_columns=train_dataset.column_names,
    num_proc=1,  # Gemma4 tokenizer may not be picklable
    desc="Adapting train set",
)
test_adapted = test_dataset.map(
    _apply_adapter,
    remove_columns=test_dataset.column_names,
    num_proc=1,
    desc="Adapting test set",
)

print(f"[OK] Train: {len(train_adapted):,} adapted samples")
print(f"[OK] Test:  {len(test_adapted):,} adapted samples")

# ---------------------------------------------------------------------------
# Preview adapted text
# ---------------------------------------------------------------------------
print(f"\n--- Adapted Sample (first 500 chars) ---")
print(train_adapted[0]["text"][:500])
print("...")

# ---------------------------------------------------------------------------
# Token length statistics
# ---------------------------------------------------------------------------
def compute_token_stats(
    dataset: Any,
    tokenizer: Any,
    text_field: str = "text",
    sample_size: int = 500,
) -> dict[str, float]:
    """Compute token-length statistics for a dataset.

    Args:
        dataset: HF dataset with a text field.
        tokenizer: Tokenizer for encoding.
        text_field: Name of the text column.
        sample_size: Number of samples to measure (for speed).

    Returns:
        Dictionary with min/max/mean/median token lengths.
    """
    n = min(sample_size, len(dataset))
    lengths: list[int] = []
    for i in range(n):
        tokens = tokenizer.encode(dataset[i][text_field], add_special_tokens=False)
        lengths.append(len(tokens))
    return {
        "samples_measured": n,
        "min_tokens": min(lengths),
        "max_tokens": max(lengths),
        "mean_tokens": statistics.mean(lengths),
        "median_tokens": statistics.median(lengths),
    }


token_stats = compute_token_stats(train_adapted, tokenizer)
print("\n--- Token Length Statistics (train) ---")
for k, v in token_stats.items():
    print(f"  {k}: {v:.0f}" if isinstance(v, float) else f"  {k}: {v}")

print(f"\n  max_length setting: {model_cfg.max_length}")
if token_stats["max_tokens"] > model_cfg.max_length:
    pct_over = sum(
        1 for i in range(min(500, len(train_adapted)))
        if len(tokenizer.encode(train_adapted[i]["text"], add_special_tokens=False)) > model_cfg.max_length
    ) / min(500, len(train_adapted)) * 100
    print(f"  [WARN] {pct_over:.1f}% of samples exceed max_length (will be truncated)")

---

## 4. Model & Training

We use **QLoRA** (Quantized Low-Rank Adaptation) to fine-tune Gemma4 efficiently:

1. **4-bit quantization** (NF4) reduces the base model from ~2.4 GB to ~0.7 GB in vRAM
2. **LoRA adapters** add only ~6M trainable parameters (0.5% of total)
3. **Gradient checkpointing** trades compute for memory

### Gemma4 Architecture Note

Gemma4 is a hybrid **state-space model** (transformer) with selective attention layers.
It requires `trust_remote_code=True` for all loading operations. The LoRA target
modules are discovered dynamically from the model's named modules -- if standard
transformer projection names (`q_proj`, `k_proj`, etc.) are not found, we fall
back to all `Linear` layers in the model.

In [ ]:
# ============================================================================
# Model Selection
# ============================================================================
# Gemma4-E2B-it is a text-only hybrid transformer model.
# Unlike the multimodal version (Gemma4-3.2B-VL), no vision encoder is loaded.

MODEL_NAME: str = "google/gemma-4-E2B-it"

print(f"[OK] Target model: {MODEL_NAME}")
print(f"[OK] Architecture: gemma4 (multimodal (text + vision))")
print(f"[OK] Parameters:   ~E2B")
print(f"[OK] Quantization: 4-bit NF4 (QLoRA)")

In [ ]:
# ============================================================================
# Load QLoRA Model
# ============================================================================

def create_qlora_model(
    model_name: str,
    bnb_config: BitsAndBytesConfig,
    attn_implementation: str = "eager",
) -> AutoModelForCausalLM:
    """Load a causal LM with 4-bit quantization for QLoRA training.

    Args:
        model_name: HuggingFace model identifier.
        bnb_config: BitsAndBytes quantization configuration.
        attn_implementation: Attention implementation (eager for Gemma4).

    Returns:
        Quantized model ready for LoRA adapter attachment.
    """
    print(f"[....] Loading {model_name} with 4-bit quantization ...")

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation=attn_implementation,
    )

    # Prepare for k-bit training (freezes base, enables gradient checkpointing)
    model = prepare_model_for_kbit_training(model)

    # Print model size info
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[OK] Total parameters:     {total_params:>12,}")
    print(f"[OK] Trainable parameters: {trainable_params:>12,}")
    print(f"[OK] Trainable %:          {trainable_params / total_params * 100:.2f}%")

    return model


model = create_qlora_model(MODEL_NAME, bnb_config, model_cfg.attn_implementation)

In [ ]:
# ============================================================================
# LoRA Configuration
# ============================================================================

def create_lora_config(lora_cfg: LoRAConfig) -> LoraConfig:
    """Create LoRA config for Gemma4.

    Gemma4 uses Gemma4ClippableLinear (inherits nn.Module, NOT nn.Linear) for
    q/k/v/o projection layers. PEFT 0.18.x doesn't recognize it as a valid
    target. Using target_modules="all-linear" resolves this because PEFT's
    _maybe_include_all_linear_layers() uses isinstance(module, nn.Linear),
    which finds the inner .linear attribute (Linear4bit, inherits nn.Linear)
    while skipping the ClippableLinear wrapper.

    This is the approach recommended by the PEFT maintainer (PR #3136) and
    verified by the community (PEFT Issue #3129).

    We also exclude vision/audio towers since we're doing text-only SFT.

    Args:
        lora_cfg: LoRA configuration dataclass.

    Returns:
        peft.LoraConfig instance.
    """
    config = LoraConfig(
        r=lora_cfg.r,
        lora_alpha=lora_cfg.lora_alpha,
        lora_dropout=lora_cfg.lora_dropout,
        target_modules=lora_cfg.target_modules,  # "all-linear"
        exclude_modules=lora_cfg.exclude_modules,  # skip vision/audio towers
        bias=lora_cfg.bias,
        task_type=lora_cfg.task_type,
    )
    print(f"[OK] LoRA config: r={lora_cfg.r}, alpha={lora_cfg.lora_alpha}")
    print(f"[OK] target_modules={lora_cfg.target_modules!r}")
    print(f"[OK] exclude_modules={lora_cfg.exclude_modules}")
    return config


peft_config = create_lora_config(lora_cfg)

In [ ]:
# ============================================================================
# Training: SFTTrainer with QLoRA
# ============================================================================

def train_model(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    train_dataset: Any,
    eval_dataset: Any,
    peft_config: LoraConfig,
    model_cfg: ModelConfig,
    train_cfg: TrainingConfig,
    smoke_test: bool = False,
) -> SFTTrainer:
    """Configure and run SFT training with QLoRA.

    Args:
        model: Quantized base model.
        tokenizer: Model tokenizer.
        train_dataset: Training dataset with "text" column.
        eval_dataset: Evaluation dataset with "text" column.
        peft_config: LoRA adapter configuration.
        model_cfg: Model configuration.
        train_cfg: Training hyperparameters.
        smoke_test: If True, limit to 5 steps.

    Returns:
        Trained SFTTrainer instance.
    """
    max_steps = 5 if smoke_test else -1

    sft_config = SFTConfig(
        output_dir=train_cfg.output_dir,

        # ---- Training hyperparameters ----
        num_train_epochs=train_cfg.num_train_epochs,
        max_steps=max_steps,
        per_device_train_batch_size=train_cfg.per_device_train_batch_size,
        per_device_eval_batch_size=train_cfg.per_device_eval_batch_size,
        gradient_accumulation_steps=train_cfg.gradient_accumulation_steps,
        learning_rate=train_cfg.learning_rate,
        weight_decay=train_cfg.weight_decay,
        warmup_ratio=train_cfg.warmup_ratio,
        lr_scheduler_type=train_cfg.lr_scheduler_type,
        optim="paged_adamw_8bit",
        seed=train_cfg.seed,

        # ---- Sequence length ----
        max_length=model_cfg.max_length,
        dataset_text_field="text",

        # ---- Precision ----
        bf16=train_cfg.bf16,
        fp16=not train_cfg.bf16 and torch.cuda.is_available(),

        # ---- Memory optimization ----
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},

        # ---- Logging ----
        logging_steps=train_cfg.logging_steps,
        logging_first_step=True,
        report_to=train_cfg.report_to,

        # ---- Saving ----
        save_strategy=train_cfg.save_strategy,

        # ---- Hub ----
        push_to_hub=False,
        hub_model_id=model_cfg.hub_model_id,

        # ---- Other ----
        remove_unused_columns=False,
        dataloader_pin_memory=True,
        dataset_num_proc=1,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        peft_config=peft_config,
    )

    # Report trainable parameters
    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in trainer.model.parameters())
    print(f"[OK] Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    if max_steps > 0:
        print(f"[SMOKE] Training for {max_steps} steps only")
    else:
        print(f"[TRAIN] Training for {train_cfg.num_train_epochs} epoch(s)")

    # ---- Train ----
    print(f"\n[....] Starting training ...")
    start_time = time.time()
    train_result = trainer.train()
    elapsed = time.time() - start_time

    # ---- Report ----
    metrics = train_result.metrics
    print(f"\n[OK] Training complete!")
    print(f"  Wall time:     {elapsed:.0f}s ({elapsed / 60:.1f} min)")
    print(f"  Train loss:    {metrics.get('train_loss', 'N/A')}")
    print(f"  Train runtime: {metrics.get('train_runtime', 'N/A')}")
    print(f"  Samples/sec:   {metrics.get('train_samples_per_second', 'N/A')}")

    return trainer


# ---- Run training ----
trainer = train_model(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_adapted,
    eval_dataset=test_adapted,
    peft_config=peft_config,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    smoke_test=SMOKE_TEST,
)

---

## 5. Inference: Testing Reasoning Traces

Now we test the trained model on three diverse prompts to verify:

1. The model generates `<think>...</think>` blocks (reasoning traces)
2. The final answer follows the thinking block
3. The reasoning is coherent and relevant to the question

We parse the output to separate the thinking and answer portions.

> **Note:** With only 5 smoke-test steps, the model will not have learned to generate
> coherent reasoning. Run a full training to see meaningful `<think>` blocks.

In [ ]:
# ============================================================================
# Inference: Test Reasoning Traces
# ============================================================================

def parse_thinking(text: str) -> tuple[str | None, str]:
    """Parse <think>...</think> block from model output.

    Args:
        text: Raw model output text.

    Returns:
        Tuple of (thinking_content, answer_content). thinking_content is None
        if no <think> block was found.
    """
    pattern = re.compile(r"<think>(.*?)</think>\s*(.*)", re.DOTALL)
    match = pattern.search(text)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    return None, text.strip()


def generate_response(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    max_new_tokens: int = 512,
) -> str:
    """Generate a response from the trained model.

    Args:
        model: The fine-tuned model (with LoRA adapter).
        tokenizer: The model tokenizer.
        prompt: User prompt to generate a response for.
        max_new_tokens: Maximum tokens to generate.

    Returns:
        Generated text (assistant response only).
    """
    messages = [{"role": "system", "content": "<|think|>"},
        {"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the generated tokens (skip the prompt)
    generated = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return generated


# ---------------------------------------------------------------------------
# Test prompts
# ---------------------------------------------------------------------------
TEST_PROMPTS: list[str] = [
    "Explain why the sky appears blue during the day but red at sunset.",
    "A farmer has 17 sheep. All but 9 run away. How many are left?",
    "What are the key differences between TCP and UDP protocols?",
]

print("[....] Running inference on test prompts ...\n")

# Put model in eval mode
trainer.model.eval()

for i, prompt in enumerate(TEST_PROMPTS):
    print(f"{'='*60}")
    print(f"Prompt {i+1}: {prompt}")
    print(f"{'='*60}")

    response = generate_response(trainer.model, tokenizer, prompt)
    thinking, answer = parse_thinking(response)

    if thinking:
        print(f"\n[THINK] (found <think> block, {len(thinking)} chars)")
        print(f"  {thinking[:300]}{'...' if len(thinking) > 300 else ''}")
        print(f"\n[ANSWER]")
        print(f"  {answer[:300]}{'...' if len(answer) > 300 else ''}")
    else:
        print(f"\n[NO THINK] No <think> block detected")
        print(f"  {answer[:400]}{'...' if len(answer) > 400 else ''}")

    print()

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print(f"\n{'='*60}")
print(f"Inference complete. {'SMOKE TEST -- reasoning quality will be poor.' if SMOKE_TEST else 'Check outputs above for reasoning quality.'}")

---

## 6. Export: Merge, GGUF Conversion & Hub Push

Final steps to make the model usable:

1. **Merge LoRA** -- fold the adapter weights back into the base model
2. **Save locally** -- full-precision merged checkpoint
3. **GGUF conversion** -- convert to llama.cpp-compatible format for local inference
4. **Push to Hub** -- upload both the merged model and GGUF files

> **Important:** GGUF conversion requires `llama-cpp-python` or the `llama.cpp`
> `convert_hf_to_gguf.py` script. Gemma4's hybrid transformer architecture may not be
> fully supported by all GGUF converters -- check compatibility first.

In [ ]:
# ============================================================================
# Merge LoRA, Save & Push to Hub
# ============================================================================
import gc
from pathlib import Path


def merge_and_export(
    trainer: SFTTrainer,
    model_cfg: ModelConfig,
    tokenizer: AutoTokenizer,
    push: bool = False,
) -> Path:
    """Merge LoRA adapter into base model and save.

    Args:
        trainer: Trained SFTTrainer with LoRA adapter.
        model_cfg: Model configuration (for hub_model_id).
        tokenizer: Tokenizer to save alongside the model.
        push: Whether to push to the Hugging Face Hub.

    Returns:
        Path to the merged model directory.
    """
    output_dir = Path(f"./{model_cfg.hub_model_id.split('/')[-1]}-merged")

    print(f"[....] Merging LoRA adapter into base model ...")

    # Merge and unload the adapter
    merged_model = trainer.model.merge_and_unload()

    # Save locally
    print(f"[....] Saving merged model to {output_dir} ...")
    merged_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"[OK] Merged model saved to {output_dir}")

    # Save the LoRA adapter separately (useful for sharing just the adapter)
    adapter_dir = Path(f"./{model_cfg.hub_model_id.split('/')[-1]}-adapter")
    trainer.model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"[OK] LoRA adapter saved to {adapter_dir}")

    # Push to Hub
    if push:
        print(f"[....] Pushing to Hub: {model_cfg.hub_model_id} ...")
        merged_model.push_to_hub(model_cfg.hub_model_id)
        tokenizer.push_to_hub(model_cfg.hub_model_id)
        print(f"[OK] Pushed to https://huggingface.co/{model_cfg.hub_model_id}")
    else:
        print(f"[SKIP] Hub push disabled (set push=True or SMOKE_TEST=False)")

    # Free memory
    del merged_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return output_dir


# ---- Run export ----
merged_dir = merge_and_export(
    trainer=trainer,
    model_cfg=model_cfg,
    tokenizer=tokenizer,
    push=not SMOKE_TEST,
)

# ---------------------------------------------------------------------------
# GGUF Conversion (optional)
# ---------------------------------------------------------------------------
# Gemma4's hybrid transformer architecture may not be supported by standard GGUF converters.
# If you have llama.cpp installed with Gemma4 support:
#
# !python llama.cpp/convert_hf_to_gguf.py \
#     {merged_dir} \
#     --outfile {merged_dir.name}.gguf \
#     --outtype q8_0
#
# Or using the Hugging Face GGUF library:
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(
#     path_or_fileobj=f"{merged_dir.name}.gguf",
#     path_in_repo=f"{merged_dir.name}-Q8_0.gguf",
#     repo_id=model_cfg.hub_model_id,
# )

print(f"\n[OK] Export pipeline complete!")

---

## Conclusion

This notebook demonstrated the full pipeline for **reasoning distillation** from
Claude Opus/Sonnet into Gemma4-E2B-it:

### What We Did

1. **Loaded** the Claude reasoning distillation dataset (10,477 training samples with 100% thinking traces)
2. **Adapted** the dataset for Gemma4 by embedding `thinking` fields as `<think>...</think>` XML blocks
3. **Fine-tuned** using QLoRA (4-bit NF4 quantization, ~6M trainable parameters)
4. **Tested** inference with three diverse prompts, parsing `<think>` blocks from output
5. **Exported** the merged model for deployment

### Next Steps

- **Full training:** Set `SMOKE_TEST = False` and run for 1-3 epochs
- **Hyperparameter tuning:** Experiment with learning rate, LoRA rank, and sequence length
- **Evaluation:** Benchmark on reasoning tasks (GSM8K, ARC, HellaSwag)
- **GRPO:** Use Group Relative Policy Optimization to further refine reasoning quality
- **Deployment:** Convert to GGUF for local inference with llama.cpp (if Gemma4 is supported)

### Model Card

| Field | Value |
|-------|-------|
| Base model | `google/gemma-4-E2B-it` |
| Architecture | `gemma4` (multimodal (text + vision)) |
| Training method | QLoRA (4-bit NF4) |
| LoRA rank | 16, alpha=32 |
| Dataset | `ermiaazarkhalili/claude-reasoning-distillation` (SFT config) |
| Training samples | 10,477 |
| Thinking format | `<think>...</think>` XML blocks |
| Hub ID | `ermiaazarkhalili/Gemma4-E2B-SFT-Claude-Reasoning` |
| Framework | TRL 1.0.0, Transformers 5.5.3, PEFT |